In [258]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from category_encoders import TargetEncoder

In [259]:
import pandas as pd
data = pd.read_csv('/Users/gopesh/Documents/Machine Learning/Bike Price predictor/final.csv')
data.columns

Index(['title', 'brand_name', 'model', 'bike_age', 'kilometer', 'fuel_type',
       'owner', 'city', 'color', 'insurance', 'mileage_owner_reported',
       'riding_range', 'cylinders', 'front_brake_type', 'rear_brake_type',
       'tyre_type', 'fuel_tank_capacity', 'reserve_fuel_capacity', 'price',
       'vehicle_type', 'is_electric'],
      dtype='object')

In [260]:
pd.set_option('display.max_columns', None)
data.head()

,title,brand_name,model,bike_age,kilometer,fuel_type,owner,city,color,insurance,mileage_owner_reported,riding_range,cylinders,front_brake_type,rear_brake_type,tyre_type,fuel_tank_capacity,reserve_fuel_capacity,price,vehicle_type,is_electric
0,2011 Hero Honda Splendor Drum,Hero Honda,Splendor Drum,15,10.819798,Not Available,First,Bhopal,Black,Comprehensive,45.0,565.65404,1.0,Not Available,Not Available,Not Available,12.0,2.581667,10000.0,bike,0
1,2021 Ducati Monster Standard,Ducati,Monster Standard,5,7.901377,Petrol,Second,Mumbai,Dark Stealth,Not Available,45.0,266.00000,2.0,Disc,Disc,Tubeless,14.0,3.500000,1050000.0,bike,0
2,2022 Bajaj Pulsar Dual Channel ABS [2022],Bajaj,Pulsar Dual Channel ABS,4,10.309286,Petrol,First,Kolkata,Brooklyn Black,Comprehensive,46.0,722.00000,1.0,Disc,Disc,Tubeless,14.0,2.800000,120000.0,bike,0
3,2019 Yamaha FZ Single Channel ABS,Yamaha,FZ Single Channel ABS,7,10.824666,Petrol,First,Chennai,BLACK,Not Available,46.0,696.00000,1.0,Disc,Drum,Tubeless,12.0,1.892000,65000.0,bike,0
4,2019 TVS Apache Carburetor ABS - Smart Xonnect,TVS,Apache Carburetor ABS - Smart Xonnect,7,10.239996,Petrol,First,Bhopal,Black,Not Available,38.0,551.00000,1.0,Disc,Disc,Tubeless,12.0,2.500000,80000.0,bike,0


In [261]:
from sklearn.model_selection import train_test_split
bike_df = data.copy()
target = 'price'

X = bike_df.drop(columns=[target])
y = bike_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [262]:
target_enc_cols = ['model']

onehot_cols = ['brand_name','fuel_type','city','front_brake_type','rear_brake_type','tyre_type','vehicle_type' ,'insurance']
ordinal_cols = ['owner']

owner_order = [['First', 'Second', 'Third', 'Fourth' , '4 or More']]

In [263]:
preprocessor = ColumnTransformer(
    transformers=[
        ('target_enc', TargetEncoder(smoothing=4, min_samples_leaf=20), target_enc_cols),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False , max_categories=20 , drop = 'first'), onehot_cols),
        ('ordinal', OrdinalEncoder(categories=owner_order , handle_unknown='use_encoded_value' , unknown_value=-1), ordinal_cols)
    ],
    remainder='passthrough'
)

In [264]:
X_train_encoded = preprocessor.fit_transform(X_train, y_train)
X_test_encoded = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_encoded = pd.DataFrame(X_train_encoded, columns=feature_names)
X_test_encoded = pd.DataFrame(X_test_encoded, columns=feature_names)

In [265]:
train_x = X_train_encoded.copy()
test_x = X_test_encoded.copy()

train_x.drop(columns=['remainder__title', 'remainder__color'], errors='ignore', inplace=True)
test_x.drop(columns=['remainder__title', 'remainder__color'], errors='ignore', inplace=True)

do feature selection on train_x and test_x.

In [266]:
import numpy as np

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

train_x = train_x.apply(pd.to_numeric, errors='coerce')
test_x = test_x.apply(pd.to_numeric, errors='coerce')

In [267]:
y_test_log.describe()

count    505.000000
mean      11.272784
std        0.716894
min        8.699681
25%       10.819798
50%       11.289794
75%       11.813037
max       13.487008
Name: price, dtype: float64

## Model building and tuning 

In [268]:
from xgboost import XGBRegressor
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

# final xgboost model with best optuna params
model = XGBRegressor(
    n_estimators=1463,
    learning_rate=0.08837925632745344,
    max_depth=5,
    subsample=0.6905008506409355,
    colsample_bytree=0.3120020888748768,
    min_child_weight=7,
    reg_alpha=0.3672208789306187,
    reg_lambda=9.683081098849447,
    gamma=0.07088132324418117,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50
)

model.fit(
    train_x, y_train_log,
    eval_set=[(test_x, y_test_log)],
    verbose=False
)

# evaluate
y_pred_log   = model.predict(test_x)
y_pred       = np.expm1(y_pred_log)

train_pred_log = model.predict(train_x)
train_pred     = np.expm1(train_pred_log)

train_mae = mean_absolute_error(y_train, train_pred)
test_mae  = mean_absolute_error(y_test, y_pred)
r2        = r2_score(y_test, y_pred)
rmse      = np.sqrt(np.mean((y_test - y_pred)**2))

print(f"Train MAE : {train_mae:,.2f}")
print(f"Test MAE  : {test_mae:,.2f}")
print(f"R2 Score  : {r2:.4f}")
print(f"RMSE      : {rmse:,.2f}")
print(f"Gap       : {test_mae - train_mae:,.2f}")

Train MAE : 13,971.45
Test MAE  : 19,454.86
R2 Score  : 0.7649
RMSE      : 36,358.84
Gap       : 5,483.40


In [269]:
# import optuna
# import numpy as np
# from xgboost import XGBRegressor
# from sklearn.metrics import mean_absolute_error
# optuna.logging.set_verbosity(optuna.logging.WARNING)  # clean output

# def objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 500, 2000),
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
#         "max_depth": trial.suggest_int("max_depth", 3, 6),
#         "subsample": trial.suggest_float("subsample", 0.5, 0.9),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8),
#         "min_child_weight": trial.suggest_int("min_child_weight", 3, 10),
#         "reg_alpha": trial.suggest_float("reg_alpha", 0.1, 5.0),
#         "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 10.0),
#         "gamma": trial.suggest_float("gamma", 0.0, 2.0),
#         "random_state": 42,
#         "n_jobs": -1,
#         "early_stopping_rounds": 50
#     }

#     model = XGBRegressor(**params)

#     model.fit(
#         train_x, y_train_log,
#         eval_set=[(test_x, y_test_log)],
#         verbose=False
#     )

#     y_pred_log = model.predict(test_x)
#     y_pred = np.expm1(y_pred_log)
#     mae = mean_absolute_error(y_test, y_pred)

#     return mae

# # run study
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=100, show_progress_bar=True)

# # results
# print("\n✅ Best Trial:")
# print(f"   MAE: {study.best_value:,.2f}")
# print(f"   Params: {study.best_params}")

In [270]:
# import optuna
# import numpy as np
# import lightgbm as lgb
# from lightgbm import LGBMRegressor
# from sklearn.metrics import mean_absolute_error, r2_score
# optuna.logging.set_verbosity(optuna.logging.WARNING)

# def objective(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 500, 2000),
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
#         "max_depth": trial.suggest_int("max_depth", 3, 7),
#         "num_leaves": trial.suggest_int("num_leaves", 20, 150),
#         "subsample": trial.suggest_float("subsample", 0.5, 0.9),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.8),
#         "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
#         "reg_alpha": trial.suggest_float("reg_alpha", 0.1, 5.0),
#         "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 10.0),
#         "random_state": 42,
#         "n_jobs": -1,
#         "verbose": -1
#     }

#     model = LGBMRegressor(**params)

#     model.fit(
#         train_x_clean, y_train_log,
#         eval_set=[(test_x_clean, y_test_log)],
#         callbacks=[
#             lgb.early_stopping(50, verbose=False),
#             lgb.log_evaluation(0)
#         ]
#     )

#     y_pred_log = model.predict(test_x)
#     y_pred = np.expm1(y_pred_log)
#     mae = mean_absolute_error(y_test, y_pred)

#     return mae

# # run study
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=300, show_progress_bar=True)

# # results
# print("\n✅ Best Trial:")
# print(f"   MAE: {study.best_value:,.2f}")
# print(f"   Params: {study.best_params}")

In [271]:
import re

def clean_col_names(df):
    df = df.copy()
    df.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in df.columns]
    return df

train_x_clean = clean_col_names(train_x)
test_x_clean  = clean_col_names(test_x)

In [272]:
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# final model with best params
lgbm_final = LGBMRegressor(
    n_estimators=1738,
    learning_rate=0.09976985436249833,
    max_depth=7,
    num_leaves=149,
    subsample=0.8321020025855697,
    colsample_bytree=0.3111060459616015,
    min_child_samples=31,
    reg_alpha=0.5538488700945204,
    reg_lambda=3.9740655375373555,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm_final.fit(
    train_x_clean, y_train_log,
    eval_set=[(test_x_clean, y_test_log)],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(0)
    ]
)

# evaluate
y_pred_log   = lgbm_final.predict(test_x_clean)
y_pred       = np.expm1(y_pred_log)

train_pred_log = lgbm_final.predict(train_x_clean)
train_pred     = np.expm1(train_pred_log)

train_mae = mean_absolute_error(y_train, train_pred)
test_mae  = mean_absolute_error(y_test, y_pred)
r2        = r2_score(y_test, y_pred)
rmse      = np.sqrt(np.mean((y_test - y_pred)**2))

print(f"Train MAE : {train_mae:,.2f}")
print(f"Test MAE  : {test_mae:,.2f}")
print(f"R2 Score  : {r2:.4f}")
print(f"RMSE      : {rmse:,.2f}")
print(f"Gap       : {test_mae - train_mae:,.2f}")

Train MAE : 13,677.75
Test MAE  : 19,712.30
R2 Score  : 0.7695
RMSE      : 36,004.10
Gap       : 6,034.55


In [273]:
# predictions from both models
xgb_pred  = np.expm1(model.predict(test_x))
lgbm_pred = np.expm1(lgbm_final.predict(test_x_clean))

# try different weightings
for xgb_w, lgbm_w in [(0.5, 0.5), (0.4, 0.6), (0.3, 0.7), (0.2, 0.8)]:
    ensemble_pred = (xgb_pred * xgb_w) + (lgbm_pred * lgbm_w)
    mae = mean_absolute_error(y_test, ensemble_pred)
    r2  = r2_score(y_test, ensemble_pred)
    print(f"XGB {xgb_w} + LGBM {lgbm_w} → MAE: {mae:,.2f}  R2: {r2:.4f}")

XGB 0.5 + LGBM 0.5 → MAE: 19,439.43  R2: 0.7696
XGB 0.4 + LGBM 0.6 → MAE: 19,467.80  R2: 0.7699
XGB 0.3 + LGBM 0.7 → MAE: 19,506.97  R2: 0.7701
XGB 0.2 + LGBM 0.8 → MAE: 19,557.73  R2: 0.7701
